# Setup — Quant Research Environment

Run this notebook first in every Colab session. It:
1. Installs Python dependencies from `requirements.txt`
2. Downloads US market data for QLib
3. Initializes QLib
4. Confirms everything is working with a quick data sanity check

In [ ]:
# Clone the repo (skip if already cloned)
import os
if not os.path.exists('quant'):
    !git clone https://github.com/JonFermin/quant.git
os.chdir('quant')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q

In [ ]:
# Download US market data for QLib
import os, zipfile, requests
from pathlib import Path
from tqdm import tqdm

data_dir = Path(os.path.expanduser("~/.qlib/qlib_data/us_data"))

if not (data_dir / "calendars").exists():
    data_dir.mkdir(parents=True, exist_ok=True)

    # QLib hosts data on GitHub releases
    base_url = "https://github.com/SunsetWolf/qlib_dataset/releases/download"
    zip_path = data_dir / "qlib_data.zip"

    # Try version-specific first, fall back to latest
    for version in ["v2/qlib_data_us_1d_0.9.7.zip", "v2/qlib_data_us_1d_latest.zip"]:
        url = f"{base_url}/{version}"
        print(f"Trying: {url}")
        resp = requests.get(url, stream=True, timeout=60)
        if resp.status_code == 200:
            print(f"Downloading ({resp.headers.get('Content-Length', 'unknown')} bytes)...")
            with open(zip_path, "wb") as f:
                for chunk in tqdm(resp.iter_content(chunk_size=8192)):
                    f.write(chunk)
            break
        print(f"  Not found (HTTP {resp.status_code}), trying next...")
    else:
        raise RuntimeError("Could not download QLib US data from any known source")

    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as zp:
        zp.extractall(str(data_dir))
    zip_path.unlink()
    print("US market data downloaded and extracted.")
else:
    print("US market data already exists, skipping download.")

!ls {data_dir}/

In [ ]:
# Initialize QLib
import qlib
from qlib.constant import REG_US

qlib.init(provider_uri="~/.qlib/qlib_data/us_data", region=REG_US)
print("QLib initialized successfully.")

In [ ]:
# Sanity check — pull a few days of close prices for AAPL
from qlib.data import D

df = D.features(
    ["AAPL"],
    fields=["$close", "$volume"],
    start_time="2024-01-01",
    end_time="2024-01-31",
)
print(f"Fetched {len(df)} rows for AAPL")
df.head(10)

---
**Setup complete.** You can now run experiment notebooks.